# Experiment: Zero-Upper-Truncated Generalized Poisson for `basket_size`

**Status: rejected — superseded by the simpler zero/upper-truncated Poisson used in the main
notebook** (`../customer_behavior.ipynb`).

The [standard Poisson experiment](standard_poisson_basket_size.ipynb) failed because it allowed
impossible basket sizes (0 or 10+). Truncating it to `[1, 9]` fixed that, but its posterior
predictive check still showed the model was **overdispersed relative to the observed data** — the
truncated Poisson's fixed mean-variance relationship ($\operatorname{Var} = E$) didn't fully match
the sharper, more peaked shape of the real basket-size distribution.

The Generalized Poisson (GP) distribution adds a dispersion parameter $\lambda$ that lets variance
differ from the mean — negative $\lambda$ produces **underdispersion**, which is what the
overdispersed-relative-to-Poisson pattern in the data called for. PyMC has no built-in truncated
Generalized Poisson, so this experiment implements one from scratch as a `pm.CustomDist`: a custom
log-probability function, a custom random sampler, and a wrapper — normalized over the truncated
support `{1, ..., 9}`.

**Why it was ultimately set aside:** the fitted dispersion parameter concentrated at essentially
zero, meaning the extra flexibility bought nothing — the data didn't actually need a dispersion
parameter beyond what Poisson already provides once truncated. Given that, the ~150 lines of custom
distribution code below add real complexity (and real risk of a subtle bug in the hand-written
`logp`/`random` methods) for zero measurable benefit over the simpler
`pm.Truncated(pm.Poisson.dist(...))` used in the main notebook. This notebook exists to document
that the more flexible model *was* tried, and why it wasn't worth keeping.

## Setup

This notebook is self-contained and can be run independently of the main notebook.

In [ ]:
import logging
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pymc as pm
import arviz as az
import arviz_plots as azp
import pytensor.tensor as pt
from scipy.special import gammaln

logging.getLogger("pymc").setLevel(logging.ERROR)
sns.set_theme(style="white")

RANDOM_SEED = 42

df = pd.read_csv("../../sql/modeling/promo_analysis_dataset.csv")
df_posterior = df.sample(n=100_000, random_state=RANDOM_SEED)
T_i = df_posterior["promotion_used"].to_numpy()
y_obs = df_posterior["basket_size"].to_numpy()

# balanced synthetic treatment vector, used only for prior predictive checks
T_prior = np.array([0] * 500 + [1] * 500)

## Custom Distribution: Zero-Upper-Truncated Generalized Poisson

The Generalized Poisson PMF, truncated and renormalized over the observed support `{1, ..., 9}`:

$$P(Y_i = k \mid \mu_i, \lambda) = \frac{\mu_i (\mu_i + \lambda k)^{k-1} e^{-\mu_i - \lambda k}}{k!}
\Big/ \sum_{j=1}^{9} \frac{\mu_i (\mu_i + \lambda j)^{j-1} e^{-\mu_i - \lambda j}}{j!}$$

Implemented below as a log-probability function (for inference) and a matching random-sampling
function (for prior/posterior predictive simulation), wrapped into a `pm.CustomDist`.

In [ ]:
def zutgp_logp(value, mu, lam):
    '''Log PMF of the zero/upper-truncated Generalized Poisson at the observed value.'''
    mu_lam_value = mu + lam * value
    logp = (
        pt.log(mu)
        + (value - 1) * pt.log(mu_lam_value)
        - mu_lam_value
        - pt.gammaln(value + 1)
    )

    # normalize over the allowed support 1,...,9
    k = pt.arange(1, 10)[:, None]
    mu_b = mu[None, :]
    mu_lam_k = mu_b + lam * k
    logp_k = (
        pt.log(mu_b)
        + (k - 1) * pt.log(mu_lam_k)
        - mu_lam_k
        - pt.gammaln(k + 1)
    )
    log_norm = pt.logsumexp(logp_k, axis=0)
    truncated_logp = logp - log_norm

    # support and Generalized Poisson parameter constraints
    valid = (
        (value >= 1) & (value <= 9)
        & (mu > 0) & (pt.abs(lam) <= 1) & (lam >= -mu / 4)
        & (mu_lam_value > 0)
    )
    return pt.switch(valid, truncated_logp, -np.inf)


def zutgp_random(mu, lam, rng=None, size=None):
    '''Draw random samples from the zero/upper-truncated Generalized Poisson via inverse CDF.'''
    if rng is None:
        rng = np.random.default_rng()

    mu, lam = np.broadcast_arrays(np.asarray(mu, dtype=float), np.asarray(lam, dtype=float))
    k = np.arange(1, 10)

    mu_lam_k = mu[..., None] + lam[..., None] * k
    logp = (
        np.log(mu[..., None])
        + (k - 1) * np.log(mu_lam_k)
        - mu_lam_k
        - gammaln(k + 1)
    )
    logp -= np.max(logp, axis=-1, keepdims=True)
    probs = np.exp(logp)
    probs /= probs.sum(axis=-1, keepdims=True)

    cdf = np.cumsum(probs, axis=-1)
    u = rng.random(mu.shape)
    return (u[..., None] > cdf).sum(axis=-1) + 1


def ZUTGeneralizedPoisson(name, mu, lam, observed=None):
    return pm.CustomDist(name, mu, lam, logp=zutgp_logp, random=zutgp_random, observed=observed)

## Model

$$Y_i \sim \text{ZUTGeneralizedPoisson}(\mu_i, \lambda), \qquad \log(\mu_i) = \alpha + \tau T_i$$

$\alpha$ and $\tau$ use the same priors selected for the truncated Poisson in the main notebook:
$\alpha \sim N(\log 4,\ 0.25)$, $\tau \sim N(0,\ 0.1)$. The new piece is $\lambda$, the dispersion
parameter. Since the truncated Poisson's posterior predictive check showed the data are
**underdispersed** relative to Poisson (variance narrower than Poisson implies), $\lambda$ is
restricted to $[-0.2, 0]$ — the negative-dispersion regime — via a truncated normal prior.

In [ ]:
def build_zutgp_model(T, y_obs=None, alpha_mu=np.log(4), alpha_sd=0.25, tau_sd=0.1, lam_sd=0.1):
    with pm.Model() as model:
        alpha = pm.Normal("alpha", mu=alpha_mu, sigma=alpha_sd)
        tau = pm.Normal("tau", mu=0, sigma=tau_sd)
        lam = pm.TruncatedNormal("lam", sigma=lam_sd, lower=-0.2, upper=0)
        mu = pm.math.exp(alpha + tau * T)
        ZUTGeneralizedPoisson("basket_size", mu=mu, lam=lam, observed=y_obs)
    return model

## Prior Selection for $\lambda$

$\alpha$ and $\tau$ keep the priors already justified for the truncated Poisson (main notebook); a
grid search here is only needed for the new dispersion parameter's spread.

In [ ]:
def run_prior_grid(model_builder, param_grid, T_prior, draws=2000, random_seed=RANDOM_SEED, **fixed_kwargs):
    results = []
    keys = list(param_grid.keys())
    for combo in product(*param_grid.values()):
        params = dict(zip(keys, combo))
        with model_builder(T_prior, y_obs=None, **fixed_kwargs, **params):
            prior_pred = pm.sample_prior_predictive(draws=draws, random_seed=random_seed)
        results.append({**params, "prior": prior_pred})
    return results


def summarize_prior_predictive(results, outcome_var, low_thresh, high_thresh):
    summary = []
    for r in results:
        y = r["prior"].prior[outcome_var].values.flatten()
        q = np.quantile(y, [0.01, 0.05, 0.50, 0.95, 0.99])
        row = {k: v for k, v in r.items() if k != "prior"}
        row.update({
            "p1": q[0], "p5": q[1], "median": q[2], "p95": q[3], "p99": q[4],
            f"prob_below_{low_thresh}": np.mean(y < low_thresh),
            f"prob_above_{high_thresh}": np.mean(y > high_thresh),
        })
        summary.append(row)
    return pd.DataFrame(summary).round(3)

In [ ]:
zutgp_prior_results = run_prior_grid(
    build_zutgp_model,
    param_grid={"lam_sd": [0.03, 0.05, 0.10]},
    T_prior=T_prior,
)
summarize_prior_predictive(zutgp_prior_results, "basket_size", low_thresh=2, high_thresh=8)

`lam_sd = 0.10` is selected: it stays centered at a basket size of 4, assigns ~10% probability to
one-item baskets, and only ~2% probability at the upper bound of 9 — giving the dispersion
parameter room to move without overriding the basket-size priors already justified for the
truncated Poisson. **Final priors:** $\alpha \sim N(\log 4,\ 0.25)$, $\tau \sim N(0,\ 0.1)$,
$\lambda \sim \text{TruncatedNormal}(0,\ 0.1,\ \text{bounds}=[-0.2, 0])$.

## Posterior

In [ ]:
with build_zutgp_model(T_i, y_obs=y_obs, lam_sd=0.1) as zutgp_model:
    zutgp_trace = pm.sample(chains=2, cores=2, random_seed=RANDOM_SEED, target_accept=0.98)

azp.plot_trace_dist(zutgp_trace, var_names=["alpha", "tau", "lam"])
plt.tight_layout()
plt.show()

display(az.summary(zutgp_trace, var_names=["alpha", "tau", "lam"], ci_prob=0.94, ci_kind="eti", round_to=3))

In [ ]:
tau_samples = zutgp_trace.posterior["tau"].values.flatten()
pct_diff = 100 * (np.exp(tau_samples) - 1)

print(f"Mean percent difference: {pct_diff.mean():.2f}%")
print(f"94% credible interval: {np.quantile(pct_diff, [0.03, 0.97]).round(2)}")
print(f"Probability difference > 0%: {np.mean(pct_diff > 0):.3f}")
print(f"Probability difference >= 5%: {np.mean(pct_diff >= 5):.4f}")

The promotion-effect conclusion is unchanged from every other model tried for this outcome — a
small, uncertain difference straddling zero. The parameter to watch here is $\lambda$: its
posterior sits right at the boundary near zero (visible in the trace plot above), which is the
first sign this extra flexibility isn't earning its keep.

## Posterior Predictive Check

In [ ]:
trace_ppc = zutgp_trace.sel(draw=slice(None, None, 20))
with zutgp_model:
    ppc = pm.sample_posterior_predictive(trace_ppc, var_names=["basket_size"], random_seed=RANDOM_SEED)

azp.plot_ppc_pava(ppc, ci_prob=0.90, var_names=["basket_size"])
plt.tight_layout()
plt.show()

azp.plot_ppc_dist(ppc, var_names=["basket_size"], num_samples=20)
plt.tight_layout()
plt.show()

In [ ]:
pred = ppc.posterior_predictive["basket_size"].values.flatten()
observed_prob = df_posterior["basket_size"].value_counts(normalize=True).sort_index()
predicted_prob = pd.Series(pred).value_counts(normalize=True).sort_index()

ppc_compare = pd.DataFrame({"observed": observed_prob, "predicted": predicted_prob}).fillna(0)
ppc_compare["difference"] = ppc_compare["predicted"] - ppc_compare["observed"]
ppc_compare.round(4)

## Why This Was Set Aside

The truncated support fix (from the [standard Poisson experiment](standard_poisson_basket_size.ipynb))
already restricts predictions to plausible basket sizes. The extra question this model was built to
answer — *does the data need a dispersion parameter beyond Poisson's fixed mean-variance link?* —
comes back **no**: $\lambda$'s posterior concentrates essentially at its zero boundary, and the
per-basket-size prediction errors above are not meaningfully smaller than the plain truncated
Poisson's. In other words, the ~150 lines of custom `logp`/`random` code here bought no measurable
improvement in fit.

**Resolution:** the main notebook uses the simpler `pm.Truncated(pm.Poisson.dist(...))` — same
truncated support, no custom distribution code, no dispersion parameter to fit or maintain. See
`zt_poisson` in `../customer_behavior.ipynb` for the model actually used to answer the business
question, and note that its posterior predictive check (rootogram + calibration plot) already
tracks the observed basket-size distribution well without this added complexity.